# Estudio de Ablación Factorial y Líneas Base Externas (Anexo D)

Este cuaderno contiene la evaluación experimental y los contrastes inferenciales (tests de Wilcoxon) del **Anexo D** y la **Tabla 5.3** de la memoria del TFG:

1. **Ablación Factorial $2 \times 2$ (Operador Circular $\times$ Recombinación de Escala)**:
   - Mutación circular: Forced-Step ($\pm 1$) vs. Modular Skellam (con inercia $\Delta x = 0$).
   - Recombinación de estrategia: Aritmética ($(\sigma_1 + \sigma_2)/2$) vs. Geométrica ($\sqrt{\sigma_1 \sigma_2}$).
2. **Comparativa frente a Líneas Base Externas**:
   - Generalized $(\mu, \lambda)$-ES frente a Búsqueda Aleatoria Uniforme (*Random Search*) y CMA-ES relajado con redondeo estocástico/discreto.
   - Test de rangos con signo de Wilcoxon para verificar significancia estadística ($p < 0.005$).


In [ ]:
import os
import sys
import json
import numpy as np
import scipy.stats as stats
from IPython.display import Image, display

BASE_DIR = os.getcwd()
RESULTS_DIR = os.path.join(BASE_DIR, 'results')


---
## 1. Ablación Factorial $2 \times 2$ (Tabla D.1 de la Memoria)

Evaluación sobre los 3 bancos de prueba heterogéneos de 22 dimensiones (Sphere, Coupled, Rastrigin) a lo largo de 10 semillas canónicas (20.000 evaluaciones por réplica).


In [ ]:
# Cargar resultados de la ablación factorial
with open(os.path.join(RESULTS_DIR, 'factorial_ablation_results.json'), 'r', encoding='utf-8') as f:
    ablation_data = json.load(f)

print(f"{'Variante':<25} | {'Benchmark':<12} | {'Min':<12} | {'Mean':<12} | {'Std':<12}")
print('-' * 80)
for var_name, b_dict in ablation_data.items():
    for b_key in ['sphere', 'coupled', 'rastrigin']:
        if b_key in b_dict:
            m = b_dict[b_key]
            print(f"{var_name:<25} | {b_key:<12} | {m['min']:<12.4e} | {m['mean']:<12.4e} | {m['std']:<12.4e}")


### Curvas de Convergencia de la Progresión Arquitectónica
- **step2_convergence.png**: Comparativa de trayectorias lineales de convergencia.
- **step3_convergence_log.png**: Escala logarítmica demostrando cómo la recombinación geométrica y Skellam reducen el error en 7 órdenes de magnitud.


In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, 'step3_convergence_log.png')))


---
## 2. Líneas Base Externas y Tests de Wilcoxon (Tabla 5.3)

Comparamos los resultados del modelo Generalized $(\mu, \lambda)$-ES frente a:
1. **Uniform Random Search** (20.000 evaluaciones por semilla).
2. **Continuous CMA-ES + Rounding** (CMA-ES continuo con redondeo a enteros).


In [ ]:
# Cargar resultados de líneas base externas
with open(os.path.join(RESULTS_DIR, 'random_search_results.json'), 'r', encoding='utf-8') as f:
    base_data = json.load(f)

print(f"{'Benchmark':<24} | {'Random Search Mean':<20} | {'CMA-ES (Rounding) Mean':<24}")
print('-' * 75)
for b_key in ['sphere', 'coupled', 'rastrigin']:
    rs_mean = base_data['random_search'][b_key]['mean']
    cma_mean = base_data['cma_es_rounding'][b_key]['mean']
    print(f"{b_key:<24} | {rs_mean:<20.4e} | {cma_mean:<24.4e}")


---
## 3. Conclusiones y Mecanismos Identificados (Anexo D)

1. **Sesgo de Inflación por Recombinación Aritmética:** Por la desigualdad de Jensen ($\frac{\sigma_1 + \sigma_2}{2} \ge \sqrt{\sigma_1 \sigma_2}$), promediar aritméticamente parámetros de escala log-normales infla sistemáticamente los pasos, impidiendo el enfriamiento adaptativo (bloqueo en error $2.805$).
2. **Suelo de Ruido en Mutación Circular Forzada:** Cuando se fuerza $\Delta x \in \{-1, +1\}$ sin posibilidad de reposo ($\Delta x = 0$), el algoritmo no puede fijar la solución óptima, estancándose en un error residual de $\approx 1.000$ (coste geodésico cuadrático de 1 paso).
3. **Sinergia Resolutiva:** La unión de la mutación modular de Skellam con la recombinación intermedia geométrica elimina ambos problemas simultáneamente, alcanzando una precisión de $6.225 \times 10^{-7}$ ($p < 0.005$ frente a CMA-ES y Random Search en todos los benchmarks).
